# Description

In this notebook, I will explore the benchmark Human Eval and using DeepSeek to generate it.

In [1]:
import os 
import sys
import numpy as np 
import pandas as pd 
import re
import io
import re
import ast
import types
import unittest
import importlib
from typing import List, Tuple, Dict, Any, Set
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import transformers

In [ ]:
model_name = "deepseek-ai/deepseek-coder-6.7b-instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(model_name,\
        trust_remote_code=True).cuda()

MAX_NEW_TOKENS = 512

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

# 1. Load data

In [3]:
PATH_CSV_DATA = "data/raw_data/human_eval.csv"

In [4]:
df = pd.read_csv(PATH_CSV_DATA)
print(f"Dataframe shape: {df.shape}")
df.sample(1)

Dataframe shape: (164, 5)


,task_id,prompt,canonical_solution,test,entry_point
42,HumanEval/42,"\n\ndef incr_list(l: list):\n """"""Return lis...",return [(e + 1) for e in l]\n,\n\nMETADATA = {}\n\n\ndef check(candidate):\n...,incr_list


In [5]:
idx = np.random.randint(0, df.shape[0])

code_description = df.loc[idx, "prompt"]
test_case = df.loc[idx, "test"]
entry_point = df.loc[idx, "entry_point"]

print("Code description:")
print(code_description)
print("=" * 20)
print("Test Case:")
print(test_case)

Code description:


def car_race_collision(n: int):
    """
    Imagine a road that's a perfectly straight infinitely long line.
    n cars are driving left to right;  simultaneously, a different set of n cars
    are driving right to left.   The two sets of cars start out being very far from
    each other.  All cars move in the same speed.  Two cars are said to collide
    when a car that's moving left to right hits a car that's moving right to left.
    However, the cars are infinitely sturdy and strong; as a result, they continue moving
    in their trajectory as if they did not collide.

    This function outputs the number of such collisions.
    """

Test Case:


METADATA = {}


def check(candidate):
    assert candidate(2) == 4
    assert candidate(3) == 9
    assert candidate(4) == 16
    assert candidate(8) == 64
    assert candidate(10) == 100




# 2. Using Deepseek to generate sample

## 2.1. Generate code

In [6]:
def extract_function(llm_text):
    # 1) Grab text between <code>...</code>
    m = re.search(r"<code>\s*(.*?)\s*</code>", llm_text, flags=re.S|re.M)
    if not m:
        raise ValueError("No <code> block found")
    code = m.group(1)

    # 2) Optionally, if the model sometimes adds backticks, strip them
    code = re.sub(r"^```(?:python)?\s*|\s*```$", "", code.strip())

    return code

In [7]:
constraints = """
Output only a complete and valid Python code for this function. 
Do not add more explanations or surrounding text and Do not change the provided function signature.
Wrap your output strictly between the markers:
<code>
... your code ...
</code>
"""

input_prompt = f"""write a complete python function
based on the following description:\n{code_description}.\n
with the following constraints:\n{constraints}
"""

print("Input prompt to:")
print(input_prompt)

Input prompt to:
write a complete python function
based on the following description:


def car_race_collision(n: int):
    """
    Imagine a road that's a perfectly straight infinitely long line.
    n cars are driving left to right;  simultaneously, a different set of n cars
    are driving right to left.   The two sets of cars start out being very far from
    each other.  All cars move in the same speed.  Two cars are said to collide
    when a car that's moving left to right hits a car that's moving right to left.
    However, the cars are infinitely sturdy and strong; as a result, they continue moving
    in their trajectory as if they did not collide.

    This function outputs the number of such collisions.
    """
.

with the following constraints:

Output only a complete and valid Python code for this function. 
Do not add more explanations or surrounding text and Do not change the provided function signature.
Wrap your output strictly between the markers:
<code>
... your cod

In [8]:
def generate_response(input_text, max_new_tokens=MAX_NEW_TOKENS):
    messages=[
        { 'role': 'system', 'content': "You are DeepSeek - a helpful assistant."},
        { 'role': 'user', 'content': f"{input_text}"}
    ]
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(model.device)
    outputs = model.generate(inputs, max_new_tokens=max_new_tokens, num_return_sequences=1, eos_token_id=tokenizer.eos_token_id)
    response = tokenizer.decode(outputs[0][len(inputs[0]):], skip_special_tokens=True)
    return response

In [9]:
output = generate_response(input_prompt)
print("Response:\n", output)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)


Response:
 <code>
def car_race_collision(n: int):
    return n * n
</code>



We can extract the complete code

In [10]:
completed_code = extract_function(output)
print(f"The complete code:\n")
print(completed_code)

The complete code:

def car_race_collision(n: int):
    return n * n


## 2.2. Evaluate the generated code

In [11]:
import builtins
import typing

def create_namespace():
    ns = {}

    # 1. Standard builtins (print, len, etc.)
    ns.update({k: getattr(builtins, k) for k in dir(builtins)})

    # 2. Install common typing names (List, Optional, etc.)
    for name in typing.__all__:
        ns[name] = getattr(typing, name)

    # 3. (Optional) Add math, random, itertools, etc.
    import math, random, itertools, statistics
    ns.update({
        'math': math,
        'random': random,
        'itertools': itertools,
        'statistics': statistics,
    })

    return ns

In [12]:
def evaluate_asserts(generated_code: str, test_code: str, entry_point: str):
    # ns = {}
    ns = create_namespace()
    
    # 1. Exec both code strings
    exec(generated_code, ns)
    exec(test_code, ns)

    candidate = ns[entry_point]     # the model's function
    check_fn = ns["check"]          # original check() function
    
    # 2. Parse the test code AST
    tree = ast.parse(test_code)

    # 3. Find the check() function body
    check_body = None
    for node in tree.body:
        if isinstance(node, ast.FunctionDef) and node.name == "check":
            check_body = node.body
            break

    if check_body is None:
        raise ValueError("check() function not found.")
    
    # 4. Evaluate each assert individually
    results = []
    for idx, stmt in enumerate(check_body):
        if isinstance(stmt, ast.Assert):
            # Convert AST back to executable code
            code = compile(ast.Module([stmt], type_ignores=[]), "<assert>", "exec")
            try:
                exec(code, {**ns, "candidate": candidate})
                results.append(("pass", None))
            except Exception as e:
                results.append(("fail", repr(e)))

    # 5. Compute pass percentage
    total = len(results)
    passed = sum(1 for r, _ in results if r == "pass")
    percentage = passed / total if total > 0 else 0.0

    return {
        "total_asserts": total,
        "passed": passed,
        "percentage": percentage,
        "detail": results
    }

In [13]:
result = evaluate_asserts(completed_code, test_case, entry_point)
print(result)

{'total_asserts': 5, 'passed': 5, 'percentage': 1.0, 'detail': [('pass', None), ('pass', None), ('pass', None), ('pass', None), ('pass', None)]}


# 3. Run through all sample

In [14]:
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FuturesTimeoutError

list_df = []

def process_idx(idx):
    # 1. Prepare input prompt
    code_description = df.loc[idx, "prompt"]
    test_case = df.loc[idx, "test"]
    entry_point = df.loc[idx, "entry_point"]

    input_prompt = f"""write a complete python function
    based on the following description:\n{code_description}.\n
    with the following constraints:\n{constraints}
    """

    output = generate_response(input_prompt)
    completed_code = extract_function(output)

    # 3. Evaluate the generated code
    result = evaluate_asserts(completed_code, test_case, entry_point)
    total_asserts = result["total_asserts"]
    passed_asserts = result["passed"]
    percentage = result["percentage"]

    return {
        "description": code_description,
        "generated_code": completed_code,
        "test_case": test_case,
        "entry_point": entry_point,
        "total_asserts": total_asserts,
        "passed_asserts": passed_asserts,
        "percentage": percentage,
    }

with ThreadPoolExecutor(max_workers=1) as executor:
    for idx in range(df.shape[0]):
        if idx % 10 == 0:
            print(f"Processing idx={idx}/{df.shape[0]}")

        future = executor.submit(process_idx, idx)

        try:
            row_result = future.result(timeout=30)  # 10-second per-index timeout
            list_df.append(row_result)
        except FuturesTimeoutError:
            print(f"Timeout at idx={idx}, skipping...")
            continue
        except Exception as e:
            print(f"Error at idx={idx}: {e}")
            continue

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Processing idx=0/164


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Processing idx=10/164


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


...

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Processing idx=150/164


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


Processing idx=160/164


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


In [15]:
output_df = pd.DataFrame(list_df)
print(f'Output dataframe shape: {output_df.shape}')
output_df.sample()

Output dataframe shape: (164, 7)


,description,generated_code,test_case,entry_point,total_asserts,passed_asserts,percentage
6,from typing import List\n\n\ndef parse_nested_...,from typing import List\n\ndef parse_nested_pa...,"\n\nMETADATA = {\n 'author': 'jt',\n 'da...",parse_nested_parens,3,3,1.0


In [16]:
# Save to CSV
output_df.to_csv("data/generated/human_eval_generated_deepseek.csv", index=False)

## 3.1. Check generated code

In [17]:
average_percentage = output_df["percentage"].mean()
print(f"Average pass percentage over all samples: {average_percentage:.2%}")    

Average pass percentage over all samples: 87.39%
